# QICK CW Calibration — Gain Curve Measurement
### RFSoC 4x2 · QICK overlay · University of Manchester / JBO
**Goal:** Sweep the QICK DAC across the RHINO science band (60–85 MHz),  
measure received power at each frequency step, and extract the frequency-  
dependent gain curve g(ν) of the signal chain.

**Hardware:** DAC_B → SMA loopback cable → ADC_D  
**Why QICK:** True direct sampling — no DDC lock, no NCO spur, no F_LO offset.  
Tone at f_DAC appears at exactly f_DAC in the ADC spectrum.

**Run all cells top-to-bottom. Each cell prints ✅ PASS or ❌ FAIL.**

---

## Cell 1 — Imports and environment check

In [ ]:
import sys, os, time, datetime
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

REQUIRED = ['numpy', 'matplotlib', 'qick']
missing  = []
for pkg in REQUIRED:
    try:
        __import__(pkg)
        print(f'  ✅ {pkg} importable')
    except ImportError:
        print(f'  ❌ {pkg} NOT FOUND')
        missing.append(pkg)

print(f'\nPython     : {sys.version.split()[0]}')
print(f'NumPy      : {np.__version__}')
print(f'Matplotlib : {matplotlib.__version__}')

if missing:
    print(f'\n❌ FAIL — missing packages: {missing}')
else:
    print('\n✅ PASS — all packages available')

## Cell 2 — Load QICK overlay
**Takes ~60 seconds. The FPGA is being programmed.**

In [ ]:
from qick import QickSoc

BIT_FILE = '/home/xilinx/qick_repo/qick_lib/qick/qick_4x2.bit'
HWH_FILE = '/home/xilinx/qick_repo/qick_lib/qick/qick_4x2.hwh'

for fpath in [BIT_FILE, HWH_FILE]:
    if os.path.exists(fpath):
        print(f'  ✅ Found: {os.path.basename(fpath)}')
    else:
        print(f'  ❌ MISSING: {fpath}')

print('\n[LOAD] Programming FPGA — please wait ~60 seconds...\n')
t0  = time.time()
soc = QickSoc(bitfile=BIT_FILE)
print(f'\n[LOAD] Done in {time.time()-t0:.0f}s')
print(soc)
print('\n✅ PASS — QICK overlay loaded')

## Cell 3 — Hardware configuration
Confirms ADC/DAC sample rates match the validated configuration.

In [ ]:
EXPECTED_ADC_FS  = 4423.680   # MHz — confirmed on board
EXPECTED_DAC_FS  = 9830.400   # MHz
FS_TOL           = 1.0         # MHz tolerance

adc_fs = soc.config['readouts'][0]['fs']
dac_fs = soc.config['gens'][0]['fs']

passed = True
for label, measured, expected in [
    ('ADC sample rate', adc_fs, EXPECTED_ADC_FS),
    ('DAC sample rate', dac_fs, EXPECTED_DAC_FS),
]:
    ok = abs(measured - expected) < FS_TOL
    symbol = '✅' if ok else '❌'
    print(f'  {symbol} {label}: {measured:.3f} MHz  (expected {expected:.3f} MHz)')
    if not ok:
        passed = False

FS_MHZ      = adc_fs
NYQUIST_MHZ = FS_MHZ / 2.0
print(f'\n  ADC Nyquist : {NYQUIST_MHZ:.2f} MHz')
print(f'  Direct sampling — no DDC, no NCO spur, no F_LO offset')
print(f'  Science band 60–85 MHz is in range: {85.0 < NYQUIST_MHZ}')
print(f'\n{"✅ PASS" if passed else "❌ FAIL"} — hardware configuration')

## Cell 4 — Experiment configuration
Edit the values in this cell to match your setup.  
All subsequent cells use these parameters — do not edit them elsewhere.

| Parameter | Loopback (bench) | Signal generator (JBO) |
|---|---|---|
| `SWEEP_START_MHZ` | 60 | 60 |
| `SWEEP_STOP_MHZ` | 85 | 85 |
| `N_STEPS` | 27 | 27 |
| `N_FRAMES_PER_STEP` | 200 | 200 |
| `N_BASELINE_FRAMES` | 500 | 500 |
| `FFT_SIZE` | 8192 | 65536 |

In [ ]:
# ── Science band sweep ────────────────────────────────────────────────────────
SWEEP_START_MHZ    = 60.0    # DAC start frequency (MHz)
SWEEP_STOP_MHZ     = 85.0    # DAC stop frequency (MHz)
N_STEPS            = 27      # number of DAC frequency steps

# ── Acquisition ───────────────────────────────────────────────────────────────
N_FRAMES_PER_STEP  = 200     # raw ADC captures per DAC step
N_BASELINE_FRAMES  = 500     # frames with DAC off for noise floor
DAC_AMPLITUDE      = 0.9     # DAC full-scale fraction (0.0–1.0)

# ── Spectral resolution ───────────────────────────────────────────────────────
# Decimated buffer: fast but limited to ~1000 samples (4.4 MHz/bin)
# DDR4 buffer:      high resolution, requires arm_ddr4 / get_ddr4 API
# For science work use DDR4 with a large FFT_SIZE
FFT_SIZE           = 8192    # points — 8192 → 540 kHz/bin; 65536 → 67 kHz/bin
WINDOW_FUNCTION    = 'hann'  # 'hann' recommended; 'rect' for Jordan's protocol

# ── Hardware channels ─────────────────────────────────────────────────────────
ADC_CH    = 0   # readout ch 0 = ADC_D SMA
DAC_GEN   = 0   # signal generator ch 0 = DAC_B SMA
# Loopback: DAC_B (gen 0) → SMA cable → ADC_D (readout 0)

# ── Output ────────────────────────────────────────────────────────────────────
SAVE_DIR  = '/home/xilinx/jupyter_notebooks/cw_calibration/'
os.makedirs(SAVE_DIR, exist_ok=True)

# ── Computed — do not edit below ──────────────────────────────────────────────
DAC_FREQS_MHZ  = np.linspace(SWEEP_START_MHZ, SWEEP_STOP_MHZ, N_STEPS)
DF_KHZ         = FS_MHZ * 1e3 / FFT_SIZE   # bin spacing

if WINDOW_FUNCTION == 'hann':
    window = np.hanning(FFT_SIZE).astype(np.float32)
elif WINDOW_FUNCTION == 'rect':
    window = np.ones(FFT_SIZE, dtype=np.float32)
else:
    window = np.blackman(FFT_SIZE).astype(np.float32)
window_power   = float(np.sum(window**2))

freq_axis_mhz  = np.fft.rfftfreq(FFT_SIZE, d=1.0/(FS_MHZ*1e6)) / 1e6  # MHz

print('--- CW calibration configuration ---')
print(f'  Sweep       : {SWEEP_START_MHZ:.1f} → {SWEEP_STOP_MHZ:.1f} MHz  ({N_STEPS} steps)')
print(f'  Step size   : {(SWEEP_STOP_MHZ-SWEEP_START_MHZ)/(N_STEPS-1):.3f} MHz')
print(f'  FFT size    : {FFT_SIZE:,}  →  {DF_KHZ:.2f} kHz/bin')
print(f'  Window      : {WINDOW_FUNCTION}')
print(f'  Frames/step : {N_FRAMES_PER_STEP}')
print(f'  Baseline    : {N_BASELINE_FRAMES} frames with DAC off')
print(f'  Save dir    : {SAVE_DIR}')
print(f'\n  Key advantage over rfsoc_sam:')
print(f'  → No DDC: tone at f_DAC appears at f_DAC in spectrum (no F_LO subtraction)')
print(f'  → No NCO spur at bin 1024')
print(f'  → Science band 60–85 MHz directly accessible')
print('\n✅ Configuration ready')

## Cell 5 — ADC capture function
Defines how to get a single raw ADC spectrum. Uses decimated buffer for speed; adapt to DDR4 for high resolution.

In [ ]:
from qick.averager_program import AveragerProgram

class CWCaptureProgram(AveragerProgram):
    """
    Minimal QICK program: arm the ADC, trigger, wait, repeat.
    No DAC pulse — the DAC tone is set externally via set_cw_tone().
    """
    def initialize(self):
        self.declare_readout(
            ch     = self.cfg['ro_ch'],
            length = self.cfg['readout_length'],
            freq   = 0,          # freq=0: no DDC shift — direct sampling
            gen_ch = None
        )
        self.synci(200)

    def body(self):
        self.trigger(
            adcs            = [self.cfg['ro_ch']],
            pins            = [0],
            adc_trig_offset = self.cfg['adc_trig_offset']
        )
        self.wait_all()
        self.sync_all(self.us2cycles(self.cfg['relax_delay']))


# Programme instance — reused for every capture
_prog_cfg = {
    'ro_ch'          : ADC_CH,
    'readout_length' : min(FFT_SIZE, 1000),  # decimated buffer max ~1000
    'adc_trig_offset': 100,
    'soft_avgs'      : 1,
    'reps'           : 1,
    'relax_delay'    : 1.0,
}
_prog = CWCaptureProgram(soc, _prog_cfg)

def capture_raw_samples():
    """
    Return FFT_SIZE raw ADC samples as float32.
    Uses the decimated buffer (fast, ~1000 samples) or DDR4 (high res).
    For FFT_SIZE <= 1000 the decimated buffer is used directly.
    For larger FFT_SIZE we use DDR4 — requires arm_ddr4 / get_ddr4.
    """
    if FFT_SIZE <= 1000:
        # Decimated buffer path (fast, limited resolution)
        iq = _prog.acquire_decimated(soc, load_pulses=False, progress=False)
        i_data = np.array(iq[0][0], dtype=np.float32)
        if len(i_data) < FFT_SIZE:
            i_data = np.pad(i_data, (0, FFT_SIZE - len(i_data)))
        return i_data[:FFT_SIZE]
    else:
        # DDR4 path (high resolution)
        SPT = 256   # samples per transfer — update from preflight Cell 7
        NT  = (FFT_SIZE // SPT) + 10
        soc.arm_ddr4(ch=ADC_CH, nt=NT)
        soc.run_rounds(_prog, rounds=1)
        raw    = soc.get_ddr4(ch=ADC_CH, nt=NT, start=None)
        i_data = raw[:, 0].astype(np.float32)
        if len(i_data) < FFT_SIZE:
            i_data = np.pad(i_data, (0, FFT_SIZE - len(i_data)))
        return i_data[:FFT_SIZE]


def compute_spectrum_db(samples):
    """
    Apply window, compute power spectrum, return in dB.
    Returns array of length FFT_SIZE//2 + 1.
    """
    windowed = samples * window
    power    = np.abs(np.fft.rfft(windowed, n=FFT_SIZE))**2 / window_power
    return (10.0 * np.log10(np.maximum(power, 1e-30))).astype(np.float32)


# Quick smoke test
print('[SMOKE TEST] Capturing one frame with no tone...')
_test = compute_spectrum_db(capture_raw_samples())
print(f'  Spectrum shape : {_test.shape}')
print(f'  Noise floor    : {float(np.median(_test)):.2f} dB')
print(f'  Peak           : {float(_test.max()):.2f} dB  at {freq_axis_mhz[int(np.argmax(_test))]:.2f} MHz')
print('\n✅ PASS — ADC capture function working')

## Cell 6 — DAC tone control function
Sets the QICK signal generator to produce a CW tone at a given frequency.

In [ ]:
def set_cw_tone(freq_mhz, amplitude=DAC_AMPLITUDE):
    """
    Configure QICK signal generator DAC_GEN to output a CW tone.

    With QICK direct sampling:
        Tone at freq_mhz -> appears at freq_mhz in ADC spectrum directly.
        No F_LO subtraction. No 2x image. No aliasing (freq_mhz < Nyquist).

    Parameters
    ----------
    freq_mhz  : float — desired tone frequency in MHz
    amplitude : float — DAC full-scale fraction (0.0–1.0)
    """
    # Convert frequency to QICK register units
    # QICK uses integer phase accumulator steps; set_gen_freq handles conversion
    soc.set_gen_freq(gen_ch=DAC_GEN, f=freq_mhz)
    soc.set_gen_amp(gen_ch=DAC_GEN, amp=int(amplitude * 32767))
    soc.set_gen_mode(gen_ch=DAC_GEN, mode='oneshot')   # continuous CW
    soc.trigger_gen(gen_ch=DAC_GEN)


def disable_tone():
    """Turn off the DAC signal generator."""
    soc.set_gen_amp(gen_ch=DAC_GEN, amp=0)
    soc.trigger_gen(gen_ch=DAC_GEN)


# Smoke test: set a tone at 70 MHz and check it appears in the spectrum
TEST_FREQ_MHZ = 70.0
print(f'[SMOKE TEST] Injecting CW tone at {TEST_FREQ_MHZ} MHz...')
set_cw_tone(TEST_FREQ_MHZ)
time.sleep(0.3)

test_spec   = compute_spectrum_db(capture_raw_samples())
expected_bin = int(TEST_FREQ_MHZ / (FS_MHZ / FFT_SIZE))
# Search in ±5 bins around expected position
search       = test_spec[max(0, expected_bin-5) : expected_bin+6]
peak_offset  = int(np.argmax(search)) - 5
actual_bin   = expected_bin + peak_offset
actual_mhz   = freq_axis_mhz[actual_bin]
snr          = float(test_spec[actual_bin]) - float(np.median(test_spec))

disable_tone()

print(f'  Expected bin  : {expected_bin} = {TEST_FREQ_MHZ:.1f} MHz')
print(f'  Actual peak   : bin {actual_bin} = {actual_mhz:.2f} MHz')
print(f'  Frequency error: {actual_mhz - TEST_FREQ_MHZ:.2f} MHz')
print(f'  SNR           : {snr:.1f} dB')

if snr > 10 and abs(actual_mhz - TEST_FREQ_MHZ) < 2.0:
    print('\n✅ PASS — CW tone visible at correct frequency, no F_LO offset needed')
else:
    print('\n❌ FAIL — tone not found or frequency offset too large')
    print('   Check: loopback cable DAC_B → ADC_D connected?')

## Cell 7 — Baseline measurement (DAC off)
Measures the noise floor spectrum with no tone injected.  
This captures any residual RFI and the ADC thermal noise.  
**Unlike rfsoc_sam, there is no NCO spur at bin 1024 here.**  
**Disconnect any antenna — loopback cable only.**

In [ ]:
print(f'[BASELINE] Measuring noise floor ({N_BASELINE_FRAMES} frames, DAC off)...')
disable_tone()
time.sleep(0.5)

baseline_acc = np.zeros(len(freq_axis_mhz), dtype=np.float64)
for k in range(N_BASELINE_FRAMES):
    baseline_acc += compute_spectrum_db(capture_raw_samples()).astype(np.float64)
    if (k+1) % 100 == 0:
        print(f'  Frame {k+1}/{N_BASELINE_FRAMES}...')

baseline = (baseline_acc / N_BASELINE_FRAMES).astype(np.float32)

noise_mean = float(np.mean(baseline))
noise_std  = float(np.std(baseline))
noise_peak = float(np.max(baseline))
peak_mhz   = float(freq_axis_mhz[int(np.argmax(baseline))])

print(f'\n--- Baseline summary ---')
print(f'  Mean noise floor : {noise_mean:.2f} dB')
print(f'  Std dev          : {noise_std:.2f} dB')
print(f'  Strongest feature: {noise_peak:.2f} dB at {peak_mhz:.1f} MHz')

# Check for NCO spur — should NOT be present with QICK
bin_1229 = int(1228.8 / (FS_MHZ / FFT_SIZE))
if bin_1229 < len(baseline):
    spur_level = float(baseline[bin_1229]) - noise_mean
    if spur_level > 10:
        print(f'\n  ⚠️  Unexpected spur at 1228.8 MHz ({spur_level:.1f} dB above noise)')
        print('     This should NOT appear with QICK — check overlay configuration')
    else:
        print(f'  ✅ No NCO spur at 1228.8 MHz (QICK direct sampling confirmed)')

np.save(f'{SAVE_DIR}cw_baseline.npy', baseline)
print(f'\n  Saved: {SAVE_DIR}cw_baseline.npy')
print('\n✅ PASS — baseline measured')

## Cell 8 — CW waterfall acquisition
Sweeps the DAC across the science band, collecting N_FRAMES_PER_STEP frames  
at each frequency. Stores the full baseline-subtracted waterfall array.  
**Jordan's pseudocode (Mar 2026):**
```
spectre = []
for i in range(n_steps):
    set_tone(f_i)
    for j in range(n_frames):
        s = acquire_spectrum
        spectre.append(s)
Spectra = np.array(spectre)
imshow(Spectra)
```

In [ ]:
N_TOTAL_FRAMES = N_STEPS * N_FRAMES_PER_STEP
print('=' * 60)
print('CW WATERFALL ACQUISITION')
print(f'  {N_STEPS} steps x {N_FRAMES_PER_STEP} frames = {N_TOTAL_FRAMES} rows')
print(f'  Sweep: {SWEEP_START_MHZ:.1f} → {SWEEP_STOP_MHZ:.1f} MHz')
print(f'  Tone formula (QICK): f_tone = f_DAC  (no F_LO offset)')
print('=' * 60)

# Allocate waterfall array
Spectra       = np.zeros((N_TOTAL_FRAMES, len(freq_axis_mhz)), dtype=np.float32)
dac_step_rows = []    # first row index for each DAC step
step_peak_snr = []    # peak SNR at each step (for gain curve)
row           = 0
t0            = time.time()

for step_idx, f_mhz in enumerate(DAC_FREQS_MHZ):

    set_cw_tone(f_mhz)
    time.sleep(0.2)   # let DAC settle
    dac_step_rows.append(row)

    step_acc = np.zeros(len(freq_axis_mhz), dtype=np.float64)

    for frame_idx in range(N_FRAMES_PER_STEP):
        raw_spec          = compute_spectrum_db(capture_raw_samples())
        subtracted        = raw_spec - baseline   # dB above noise floor
        Spectra[row]      = subtracted
        step_acc         += subtracted.astype(np.float64)
        row              += 1

    # Expected tone bin for this step
    expected_bin = int(f_mhz / (FS_MHZ / FFT_SIZE))
    expected_bin = min(expected_bin, len(freq_axis_mhz) - 1)

    step_avg = (step_acc / N_FRAMES_PER_STEP).astype(np.float32)

    # Search ±5 bins around expected position
    lo = max(0, expected_bin - 5)
    hi = min(len(step_avg), expected_bin + 6)
    peak_snr = float(np.max(step_avg[lo:hi]))
    step_peak_snr.append(peak_snr)

    elapsed = time.time() - t0
    print(f'  Step {step_idx+1:3d}/{N_STEPS}  '
          f'DAC={f_mhz:.2f} MHz  '
          f'rows {dac_step_rows[-1]}–{row-1}  '
          f'tone_SNR={peak_snr:.1f} dB  '
          f'elapsed={elapsed:.1f}s')

disable_tone()
total_time = time.time() - t0
dt_ms      = total_time / N_TOTAL_FRAMES * 1000

print(f'\n  Done. {row} rows in {total_time:.1f}s  (ΔT = {dt_ms:.2f} ms/frame)')

# Save
ts = datetime.datetime.utcnow().strftime('%Y%m%d_%H%M%S')
np.save(f'{SAVE_DIR}cw_waterfall_{ts}.npy',       Spectra)
np.save(f'{SAVE_DIR}cw_dac_step_rows_{ts}.npy',   np.array(dac_step_rows))
np.save(f'{SAVE_DIR}cw_dac_freqs_{ts}.npy',       DAC_FREQS_MHZ)
np.save(f'{SAVE_DIR}cw_freq_axis_{ts}.npy',       freq_axis_mhz)
np.save(f'{SAVE_DIR}cw_step_peak_snr_{ts}.npy',   np.array(step_peak_snr))
print(f'\n  Saved: cw_waterfall_{ts}.npy  (shape {Spectra.shape})')
print('\n✅ PASS — waterfall acquired')

## Cell 9 — Waterfall plot
Full baseline-subtracted waterfall with expected tone positions marked.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 9),
                         gridspec_kw={'width_ratios': [3, 1]})

ax, ax2 = axes

# ── Left: waterfall imshow ────────────────────────────────────────────────────
n_rows  = Spectra.shape[0]
vmin    = -5.0
vmax    = float(np.nanpercentile(Spectra, 99.5))
vmax    = max(vmax, 10.0)

# Build frequency extent in MHz
f_min_mhz = float(freq_axis_mhz[0])
f_max_mhz = float(freq_axis_mhz[-1])

im = ax.imshow(
    Spectra,
    aspect='auto',
    extent=[f_min_mhz, f_max_mhz, n_rows, 0],
    vmin=vmin, vmax=vmax,
    cmap='viridis',
    interpolation='nearest',
    origin='upper'
)

# Mark step boundaries and expected tone positions
for i, r in enumerate(dac_step_rows):
    ax.axhline(r, color='white', lw=0.4, alpha=0.4, linestyle='--')
    tone_mhz = float(DAC_FREQS_MHZ[i])
    ax.axvline(tone_mhz, color='red', lw=0.8, alpha=0.6)
    if i % 5 == 0:
        ax.text(tone_mhz + (SWEEP_STOP_MHZ - SWEEP_START_MHZ)*0.01,
                r + N_FRAMES_PER_STEP * 0.5,
                f'{tone_mhz:.1f} MHz',
                fontsize=7, color='white', va='center',
                bbox=dict(facecolor='darkred', alpha=0.55, pad=1))

# Shade the science band
ax.axvspan(SWEEP_START_MHZ, SWEEP_STOP_MHZ, alpha=0.06, color='cyan')

ax.set_xlim(max(0, SWEEP_START_MHZ - 5), SWEEP_STOP_MHZ + 5)
ax.set_xlabel(
    f'Frequency (MHz)  [{DF_KHZ:.2f} kHz/bin  |  '
    f'tone at f_DAC directly (QICK direct sampling)]',
    fontsize=10
)
ax.set_ylabel('Frame index  (time,  each row = one get_frame())', fontsize=10)
ax.set_title(
    f'QICK CW Waterfall  (baseline subtracted)\n'
    f'Sweep: {SWEEP_START_MHZ:.0f}–{SWEEP_STOP_MHZ:.0f} MHz  |  '
    f'{N_FRAMES_PER_STEP} frames/step  |  {N_STEPS} steps  |  '
    f'colour = dB above noise floor',
    fontsize=10
)
plt.colorbar(im, ax=ax, label='dB above noise floor', shrink=0.8)

# ── Right: gain curve (peak tone power per step) ──────────────────────────────
steps = np.arange(N_STEPS)
ax2.barh(steps, step_peak_snr, color='steelblue', alpha=0.8, height=0.7)
ax2.set_yticks(steps)
ax2.set_yticklabels([f'{f:.1f}' for f in DAC_FREQS_MHZ], fontsize=7)
ax2.set_xlabel('Peak tone SNR\n(dB above noise floor)', fontsize=9)
ax2.set_ylabel('DAC frequency (MHz)', fontsize=9)
ax2.set_title('Preliminary\ngain curve g(ν)', fontsize=10)
ax2.axvline(0, color='grey', lw=0.8, linestyle=':')
ax2.grid(True, alpha=0.3, axis='x')
ax2.invert_yaxis()

plt.suptitle(
    f'RHINO RFSoC 4x2 — CW Calibration  (QICK overlay)\n'
    f'Science band: {SWEEP_START_MHZ:.0f}–{SWEEP_STOP_MHZ:.0f} MHz  |  '
    f'{ts}',
    fontsize=11
)
plt.tight_layout()
out = f'{SAVE_DIR}cw_waterfall_plot_{ts}.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'\n  Saved: {out}')
print('\n✅ PASS — waterfall plot generated')

## Cell 10 — Averaged spectrum per step
Averages all N_FRAMES_PER_STEP frames at each DAC step.  
Produces a clean 27-row image showing the tone at centre of each row.  
This is the publishable result for the RASTI paper.

In [ ]:
# Build N_STEPS-row averaged image (one row per DAC step)
n_freq   = len(freq_axis_mhz)
avg_img  = np.zeros((N_STEPS, n_freq), dtype=np.float32)

for i, r_start in enumerate(dac_step_rows):
    r_end      = r_start + N_FRAMES_PER_STEP
    block      = Spectra[r_start:r_end]
    avg_img[i] = np.mean(block, axis=0)

vmin_avg = -5.0
vmax_avg = float(np.nanpercentile(avg_img, 99))
vmax_avg = max(vmax_avg, 10.0)

fig, axes = plt.subplots(1, 2, figsize=(14, 8),
                         gridspec_kw={'width_ratios': [3, 1]})
ax, ax2 = axes

# ── Left: clean averaged waterfall ───────────────────────────────────────────
img_extent = [f_min_mhz, f_max_mhz, N_STEPS - 0.5, -0.5]
im = ax.imshow(
    avg_img,
    aspect='auto',
    extent=img_extent,
    vmin=vmin_avg, vmax=vmax_avg,
    cmap='viridis',
    interpolation='nearest',
    origin='upper'
)

# Mark expected tone position as red dashed line at the actual DAC frequency
for i, f_mhz in enumerate(DAC_FREQS_MHZ):
    ax.axvline(f_mhz, color='red', lw=0.6, alpha=0.5)

ax.set_yticks(range(N_STEPS))
ax.set_yticklabels([f'{f:.1f} MHz' for f in DAC_FREQS_MHZ], fontsize=7)
ax.set_xlim(max(0, SWEEP_START_MHZ - 5), SWEEP_STOP_MHZ + 5)
ax.set_xlabel(
    f'Frequency (MHz)  [{DF_KHZ:.2f} kHz/bin  |  '
    f'tone = f_DAC  (QICK direct sampling, no offset)]',
    fontsize=10
)
ax.set_ylabel('DAC frequency step', fontsize=10)
ax.set_title(
    f'Averaged CW Waterfall — {N_FRAMES_PER_STEP} frames/step\n'
    f'Red lines = expected tone position  |  colour = dB above noise floor',
    fontsize=10
)
plt.colorbar(im, ax=ax, label='dB above noise floor', shrink=0.8)

# ── Right: gain curve ────────────────────────────────────────────────────────
# Peak power in ±3-bin window around expected tone bin
gain_curve = np.zeros(N_STEPS, dtype=np.float32)
for i, f_mhz in enumerate(DAC_FREQS_MHZ):
    expected_bin = int(f_mhz / (FS_MHZ / FFT_SIZE))
    lo = max(0, expected_bin - 3)
    hi = min(n_freq, expected_bin + 4)
    gain_curve[i] = float(np.max(avg_img[i, lo:hi]))

steps = np.arange(N_STEPS)
ax2.barh(steps, gain_curve, color='steelblue', alpha=0.85, height=0.7)
ax2.set_yticks(steps)
ax2.set_yticklabels([f'{f:.1f}' for f in DAC_FREQS_MHZ], fontsize=7)
ax2.set_xlabel('Tone power g(ν)\n(dB above noise floor)', fontsize=9)
ax2.set_title('Gain curve g(ν)\n(peak in ±3-bin window)', fontsize=10)
ax2.axvline(0, color='grey', lw=0.8, linestyle=':')
ax2.grid(True, alpha=0.3, axis='x')
ax2.invert_yaxis()

plt.suptitle(
    f'RHINO RFSoC 4x2 — CW Calibration Gain Curve  (QICK)\n'
    f'Science band {SWEEP_START_MHZ:.0f}–{SWEEP_STOP_MHZ:.0f} MHz  |  '
    f'{N_STEPS} steps  |  {N_FRAMES_PER_STEP} frames averaged per step',
    fontsize=11
)
plt.tight_layout()
out2 = f'{SAVE_DIR}cw_gain_curve_{ts}.png'
plt.savefig(out2, dpi=150, bbox_inches='tight')
plt.show()

# Also save the gain curve as a numpy array for the RASTI paper
np.save(f'{SAVE_DIR}cw_gain_curve_{ts}.npy',    gain_curve)
np.save(f'{SAVE_DIR}cw_avg_waterfall_{ts}.npy', avg_img)
print(f'\n  Saved: {out2}')
print(f'  Saved: {SAVE_DIR}cw_gain_curve_{ts}.npy')
print('\n✅ PASS — averaged waterfall and gain curve generated')

## Cell 11 — Tone SNR summary table
Prints a summary table of the measured tone SNR at each DAC step.  
This is the key diagnostic — each step should show clear positive SNR  
at the expected frequency.

In [ ]:
print('=' * 70)
print('CW CALIBRATION RESULTS — TONE SNR PER STEP')
print(f'  Sweep: {SWEEP_START_MHZ:.1f}–{SWEEP_STOP_MHZ:.1f} MHz  |  '
      f'{N_FRAMES_PER_STEP} frames/step  |  {N_STEPS} steps')
print(f'  Tone formula (QICK): f_tone = f_DAC  (no F_LO offset)')
print('=' * 70)
print(f'  {"Step":>5}  {"DAC (MHz)":>10}  {"Exp. bin":>9}  '
      f'{"Peak bin":>9}  {"Peak MHz":>10}  {"SNR (dB)":>10}  {"Pass?"}')
print('  ' + '-' * 68)

n_pass = 0
for i, f_mhz in enumerate(DAC_FREQS_MHZ):
    expected_bin = int(f_mhz / (FS_MHZ / FFT_SIZE))
    expected_bin = min(expected_bin, n_freq - 1)

    lo = max(0, expected_bin - 5)
    hi = min(n_freq, expected_bin + 6)
    peak_offset = int(np.argmax(avg_img[i, lo:hi]))
    actual_bin  = lo + peak_offset
    actual_mhz  = float(freq_axis_mhz[actual_bin])
    snr         = float(avg_img[i, actual_bin])
    passed      = snr > 6.0 and abs(actual_mhz - f_mhz) < 2.0

    if passed:
        n_pass += 1

    sym = '✅' if passed else '❌'
    print(f'  {i+1:>5d}  {f_mhz:>10.2f}  {expected_bin:>9d}  '
          f'{actual_bin:>9d}  {actual_mhz:>10.2f}  {snr:>10.1f}  {sym}')

print('  ' + '-' * 68)
print(f'\n  Pass rate: {n_pass}/{N_STEPS} steps with SNR > 6 dB and frequency error < 2 MHz')
print()
if n_pass == N_STEPS:
    print('  ✅ ALL STEPS PASS — gain curve complete')
elif n_pass > N_STEPS * 0.8:
    print(f'  ⚠️  {N_STEPS - n_pass} steps failed — check for RFI at those frequencies')
else:
    print(f'  ❌ {N_STEPS - n_pass} steps failed — check loopback cable and DAC amplitude')

## Cell 12 — Save all results
Saves the complete dataset and prints download instructions.  
This is everything needed for the RASTI paper results section.

In [ ]:
print('[SAVE] Saving final results...')

files_saved = {
    f'cw_waterfall_{ts}.npy'     : 'Full waterfall array (N_TOTAL_FRAMES × n_bins)',
    f'cw_avg_waterfall_{ts}.npy' : 'Averaged waterfall (N_STEPS × n_bins)',
    f'cw_gain_curve_{ts}.npy'    : 'Gain curve g(ν) — one value per DAC step',
    f'cw_dac_freqs_{ts}.npy'     : 'DAC frequencies (MHz) — N_STEPS values',
    f'cw_freq_axis_{ts}.npy'     : 'Frequency axis (MHz) — n_bins values',
    f'cw_baseline.npy'           : 'Noise floor baseline (n_bins values)',
    f'cw_waterfall_plot_{ts}.png': 'Full waterfall plot',
    f'cw_gain_curve_{ts}.png'    : 'Averaged waterfall + gain curve plot',
}

for fname, desc in files_saved.items():
    fpath = f'{SAVE_DIR}{fname}'
    if os.path.exists(fpath):
        size_mb = os.path.getsize(fpath) / 1e6
        print(f'  ✅ {fname:<40} ({size_mb:.1f} MB)  — {desc}')
    else:
        print(f'  ⚠️  {fname:<40} NOT FOUND')

print(f'\n--- Reload these results on any machine ---')
print(f'import numpy as np')
print(f'Spectra    = np.load("{SAVE_DIR}cw_waterfall_{ts}.npy")')
print(f'avg_img    = np.load("{SAVE_DIR}cw_avg_waterfall_{ts}.npy")')
print(f'gain_curve = np.load("{SAVE_DIR}cw_gain_curve_{ts}.npy")')
print(f'dac_freqs  = np.load("{SAVE_DIR}cw_dac_freqs_{ts}.npy")')
print(f'freq_axis  = np.load("{SAVE_DIR}cw_freq_axis_{ts}.npy")')
print()
print(f'--- Download from board ---')
print(f'scp xilinx@192.168.2.99:{SAVE_DIR}*_{ts}* .')
print()
print('✅ DONE — CW calibration experiment complete')
print()
print('Next steps:')
print('  1. Vary DAC amplitude (0.1–1.0) at each frequency step')
print('     to extract g(ν) via the linear fit P_ADC = g(ν)·P_CW + P_noise')
print('  2. Replace loopback with signal generator for absolute power reference')
print('  3. Compare gain curve with rfsoc_sam result (where available)')